In [ ]:
# file: notebooks/01_eda_and_visualization.ipynb
# CELL 1: Import Libraries
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_style("whitegrid")


In [ ]:
# CELL 2: Load and Combine Data
# This cell combines the four raw CSV files into a single DataFrame for analysis.
RAW_DATA_PATH = '../data/raw/'
files_to_load = ['blackhole.csv', 'dodag.csv', 'flooding.csv', 'rank.csv']
dataframes = []

for file_name in files_to_load:
    file_path = os.path.join(RAW_DATA_PATH, file_name)
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        dataframes.append(df)
        print(f"Loaded {len(df)} rows from {file_name}")
    else:
        print(f"File not found: {file_path}")

# Combine all dataframes, then separate normal and attack traffic to handle duplicates
all_data = pd.concat(dataframes, ignore_index=True)
normal_df = all_data[all_data['category'] == 'Normal'].drop_duplicates()
attack_df = all_data[all_data['category']!= 'Normal']
df_combined = pd.concat([normal_df, attack_df], ignore_index=True)

print(f"\nCombined dataset created with {len(df_combined)} total rows.")


In [ ]:
# CELL 3: Initial Data Inspection
# Get a quick overview of the data types and check for missing values.
print("--- DataFrame Info ---")
df_combined.info()

print("\n--- First 5 Rows ---")
display(df_combined.head())

print("\n--- Summary Statistics ---")
display(df_combined.describe())


In [ ]:
# CELL 4: Visualize Class Distribution
# Check if the dataset is balanced across the different categories.
plt.figure(figsize=(10, 6))
sns.countplot(x='category', data=df_combined, order=df_combined['category'].value_counts().index)
plt.title('Distribution of Attack Categories')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# CELL 5: Visualize Feature Distributions
# Plot histograms for all numeric features to understand their distributions.
# This helps identify skewed data or features with low variance.
numeric_cols = df_combined.select_dtypes(include=np.number).columns.tolist()
# We can exclude 'label' as it's just a mirror of the category.
numeric_cols.remove('label')

df_combined[numeric_cols].hist(bins=30, figsize=(20, 15), layout=(-1, 4))
plt.suptitle('Histograms of Numeric Features', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# CELL 6: Explore Feature Correlations
# A correlation matrix helps us understand the linear relationships between features.
# High correlation between features can sometimes be problematic for models.
plt.figure(figsize=(18, 15))
correlation_matrix = df_combined[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm')
plt.title('Correlation Matrix of Numeric Features')
plt.show()
